# **Временные ряды (проект)**

### Предварительная загрузка библиотек, и импортирование

In [1]:
!pip install -r requirements.txt

In [2]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

import sys
sys.path.append('src')

from prepare_data import prepare_data
from naive import naive_forecast
from metrics import metrics
from catboost_model import catboost_recursive, catboost_direct
from PatchTST_model import PatchTST_model

### Обработка датасета

In [3]:
df = pd.read_csv('dataset/kalimati_tarkari_dataset.csv')

In [4]:
train, test_1, test_14, test_30 = prepare_data(df)

### Наивный прогноз

In [5]:
df_naive_1 = naive_forecast(train, test_1, 1)
df_naive_14 = naive_forecast(train, test_14, 14)
df_naive_30 = naive_forecast(train, test_30, 30)

In [6]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=train[train['Commodity'] == 'Mushroom(Kanya)']['Date'], y=train[train['Commodity'] == 'Mushroom(Kanya)']['Average'], name='train'))
fig.add_trace(go.Scatter(x=df_naive_30[df_naive_30['Commodity'] == 'Mushroom(Kanya)']['Date'], y=df_naive_30[df_naive_30['Commodity'] == 'Mushroom(Kanya)']['Average'], name='Реальная цена'))
fig.add_trace(go.Scatter(x=df_naive_30[df_naive_30['Commodity'] == 'Mushroom(Kanya)']['Date'], y=df_naive_30[df_naive_30['Commodity'] == 'Mushroom(Kanya)']['naive'], name='Наивный прогноз'))

fig.update_layout(
    title="Пример - Mushroom(Kanya) (30 дней), наивный прогноз",
    xaxis_title="Дата",
    yaxis_title="Цена",
)

fig.show()

In [7]:
print('Метрики для наивного прогноза (1 день):', metrics(df_naive_1, true_col='Average', pred_col='naive'))
print('Метрики для наивного прогноза (14 дней):', metrics(df_naive_14, true_col='Average', pred_col='naive'))
print('Метрики для наивного прогноза (30 дней):', metrics(df_naive_30, true_col='Average', pred_col='naive'))

Метрики для наивного прогноза (1 день): {'MAE': np.float64(4.534883720930233), 'RMSE': np.float64(9.970887856713803), 'SMAPE': np.float64(0.06020491940824579)}
Метрики для наивного прогноза (14 дней): {'MAE': np.float64(9.997508305647841), 'RMSE': np.float64(17.761583184201296), 'SMAPE': np.float64(0.1225935326440409)}
Метрики для наивного прогноза (30 дней): {'MAE': np.float64(15.997286821705426), 'RMSE': np.float64(26.50505656013233), 'SMAPE': np.float64(0.20882604223878695)}


### Catboost (recursive и direct)


#### Catboost recursive


In [8]:
test_1_catboost = catboost_recursive(train, df_naive_1, 1)
test_14_catboost = catboost_recursive(train, df_naive_14, 14)
test_30_catboost = catboost_recursive(train, df_naive_30, 30)

Learning rate set to 0.087418
0:	learn: 72.0606638	total: 95ms	remaining: 1m 34s
1:	learn: 66.5318167	total: 137ms	remaining: 1m 8s
2:	learn: 61.4520066	total: 167ms	remaining: 55.6s
3:	learn: 56.8295258	total: 204ms	remaining: 50.7s
4:	learn: 52.6586457	total: 233ms	remaining: 46.4s
5:	learn: 48.7722628	total: 276ms	remaining: 45.7s
6:	learn: 45.3060454	total: 299ms	remaining: 42.4s
7:	learn: 42.0713945	total: 335ms	remaining: 41.6s
8:	learn: 39.1747348	total: 383ms	remaining: 42.2s
9:	learn: 36.5228396	total: 440ms	remaining: 43.6s
10:	learn: 34.1132566	total: 478ms	remaining: 43s
11:	learn: 31.9234720	total: 516ms	remaining: 42.5s
12:	learn: 29.9627437	total: 556ms	remaining: 42.2s
13:	learn: 28.1992319	total: 588ms	remaining: 41.4s
14:	learn: 26.5731089	total: 619ms	remaining: 40.7s
15:	learn: 25.1451487	total: 663ms	remaining: 40.8s
16:	learn: 23.8462689	total: 697ms	remaining: 40.3s
17:	learn: 22.7150162	total: 750ms	remaining: 40.9s
18:	learn: 21.6880969	total: 791ms	remaining: 

In [9]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=train[train['Commodity'] == 'Mushroom(Kanya)']['Date'], y=train[train['Commodity'] == 'Mushroom(Kanya)']['Average'], name='train'))
fig.add_trace(go.Scatter(x=test_30_catboost[test_30_catboost['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_catboost[test_30_catboost['Commodity'] == 'Mushroom(Kanya)']['Average'], name='Реальная цена'))
fig.add_trace(go.Scatter(x=test_30_catboost[test_30_catboost['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_catboost[test_30_catboost['Commodity'] == 'Mushroom(Kanya)']['naive'], name='Наивный прогноз'))
fig.add_trace(go.Scatter(x=test_30_catboost[test_30_catboost['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_catboost[test_30_catboost['Commodity'] == 'Mushroom(Kanya)']['catboost_recursive'], name='catboost recursive'))

fig.update_layout(
    title="Пример - Mushroom(Kanya) (30 дней), наивный прогноз + catboost recursive",
    xaxis_title="Дата",
    yaxis_title="Цена",
)

fig.show()

In [10]:
print('Метрики для catboost recursive прогноза (1 день):', metrics(test_1_catboost, true_col='Average', pred_col='catboost_recursive'))
print('Метрики для catboost recursive прогноза (14 дней):', metrics(test_14_catboost, true_col='Average', pred_col='catboost_recursive'))
print('Метрики для catboost recursive прогноза (30 дней):', metrics(test_30_catboost, true_col='Average', pred_col='catboost_recursive'))

Метрики для catboost recursive прогноза (1 день): {'MAE': np.float64(5.215836988331557), 'RMSE': np.float64(9.829619766642734), 'SMAPE': np.float64(0.06887975826574523)}
Метрики для catboost recursive прогноза (14 дней): {'MAE': np.float64(10.80468918690851), 'RMSE': np.float64(17.840776896388032), 'SMAPE': np.float64(0.13309396129013387)}
Метрики для catboost recursive прогноза (30 дней): {'MAE': np.float64(16.566986975874883), 'RMSE': np.float64(25.603283592711172), 'SMAPE': np.float64(0.21990663782518818)}


#### Catboost direct

In [11]:
test_1_catboost_new = catboost_direct(train, test_1_catboost, 1)
test_14_catboost_new = catboost_direct(train, test_14_catboost, 14)
test_30_catboost_new = catboost_direct(train, test_30_catboost, 30)

Выходные данные были обрезаны до нескольких последних строк (5000).
4:	learn: 56.9419121	total: 114ms	remaining: 22.7s
5:	learn: 53.9173137	total: 133ms	remaining: 22.1s
6:	learn: 51.2726954	total: 152ms	remaining: 21.6s
7:	learn: 48.8480284	total: 171ms	remaining: 21.2s
8:	learn: 46.7397824	total: 192ms	remaining: 21.2s
9:	learn: 44.8933940	total: 216ms	remaining: 21.4s
10:	learn: 43.2676350	total: 235ms	remaining: 21.1s
11:	learn: 41.8381752	total: 255ms	remaining: 21s
12:	learn: 40.5840356	total: 274ms	remaining: 20.8s
13:	learn: 39.4932687	total: 293ms	remaining: 20.7s
14:	learn: 38.5351210	total: 312ms	remaining: 20.5s
15:	learn: 37.7149918	total: 332ms	remaining: 20.4s
16:	learn: 36.9859755	total: 350ms	remaining: 20.2s
17:	learn: 36.3837085	total: 368ms	remaining: 20.1s
18:	learn: 35.8355212	total: 390ms	remaining: 20.1s
19:	learn: 35.3791410	total: 415ms	remaining: 20.3s
20:	learn: 34.9390119	total: 433ms	remaining: 20.2s
21:	learn: 34.5858771	total: 452ms	remaining: 20.1s
22:	

In [12]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=train[train['Commodity'] == 'Mushroom(Kanya)']['Date'], y=train[train['Commodity'] == 'Mushroom(Kanya)']['Average'], name='train'))
fig.add_trace(go.Scatter(x=test_30_catboost_new[test_30_catboost_new['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_catboost_new[test_30_catboost_new['Commodity'] == 'Mushroom(Kanya)']['Average'], name='Реальная цена'))
fig.add_trace(go.Scatter(x=test_30_catboost_new[test_30_catboost_new['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_catboost_new[test_30_catboost_new['Commodity'] == 'Mushroom(Kanya)']['naive'], name='Наивный прогноз'))
fig.add_trace(go.Scatter(x=test_30_catboost_new[test_30_catboost_new['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_catboost_new[test_30_catboost_new['Commodity'] == 'Mushroom(Kanya)']['catboost_recursive'], name='catboost recursive'))
fig.add_trace(go.Scatter(x=test_30_catboost_new[test_30_catboost_new['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_catboost_new[test_30_catboost_new['Commodity'] == 'Mushroom(Kanya)']['catboost_direct'], name='ccatboost direct'))

fig.update_layout(
    title="Пример - Mushroom(Kanya) (30 дней), наивный прогноз + catboost recursive",
    xaxis_title="Дата",
    yaxis_title="Цена",
)

fig.show()

In [13]:
print('Метрики для catboost recursive прогноза (1 день):', metrics(test_1_catboost_new, true_col='Average', pred_col='catboost_direct'))
print('Метрики для catboost recursive прогноза (14 дней):', metrics(test_14_catboost_new, true_col='Average', pred_col='catboost_direct'))
print('Метрики для catboost recursive прогноза (30 дней):', metrics(test_30_catboost_new, true_col='Average', pred_col='catboost_direct'))

Метрики для catboost recursive прогноза (1 день): {'MAE': np.float64(5.215836988331557), 'RMSE': np.float64(9.829619766642734), 'SMAPE': np.float64(0.06887975826574523)}
Метрики для catboost recursive прогноза (14 дней): {'MAE': np.float64(10.841549861536265), 'RMSE': np.float64(17.72905609945765), 'SMAPE': np.float64(0.13288793509657562)}
Метрики для catboost recursive прогноза (30 дней): {'MAE': np.float64(17.256390019161692), 'RMSE': np.float64(26.348608219093695), 'SMAPE': np.float64(0.22908549228717343)}


### PatchTST_model

In [14]:
test_1_PatchTST_model = PatchTST_model(train, test_1_catboost_new, 1)
test_14_PatchTST_model = PatchTST_model(train, test_14_catboost_new, 14)
test_30_PatchTST_model = PatchTST_model(train, test_30_catboost_new, 30)

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE               │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d     │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm      │      0 │ train │     0 │
│ 3 │ model        │ PatchTST_backbone │  400 K │ train │     0 │
└───┴──────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 400 K                                                                                            
Non-trainable params: 3                                                                                            
Total params: 400 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 90                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.



In [17]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=train[train['Commodity'] == 'Mushroom(Kanya)']['Date'], y=train[train['Commodity'] == 'Mushroom(Kanya)']['Average'], name='train'))
fig.add_trace(go.Scatter(x=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['Average'], name='Реальная цена'))
fig.add_trace(go.Scatter(x=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['naive'], name='Наивный прогноз'))
fig.add_trace(go.Scatter(x=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['catboost_recursive'], name='catboost recursive'))
fig.add_trace(go.Scatter(x=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['catboost_direct'], name='catboost direct'))
fig.add_trace(go.Scatter(x=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['Date'], y=test_30_PatchTST_model[test_30_PatchTST_model['Commodity'] == 'Mushroom(Kanya)']['PatchTST_model'], name='PatchTST model'))

fig.update_layout(
    title="Пример - Mushroom(Kanya) (30 дней), наивный прогноз + catboost recursive + PatchTST",
    xaxis_title="Дата",
    yaxis_title="Цена",
)

fig.show()

In [16]:
print('Метрики для catboost recursive прогноза (1 день):', metrics(юtest_1_PatchTST_model, true_col='Average', pred_col='PatchTST_model'))
print('Метрики для catboost recursive прогноза (14 дней):', metrics(test_14_PatchTST_model, true_col='Average', pred_col='PatchTST_model'))
print('Метрики для catboost recursive прогноза (30 дней):', metrics(test_30_PatchTST_model, true_col='Average', pred_col='PatchTST_model'))

Метрики для catboost recursive прогноза (1 день): {'MAE': np.float64(4.645643189895985), 'RMSE': np.float64(10.033251706453363), 'SMAPE': np.float64(0.061577259392437037)}
Метрики для catboost recursive прогноза (14 дней): {'MAE': np.float64(9.997261418060607), 'RMSE': np.float64(17.760913798597073), 'SMAPE': np.float64(0.12260949698835778)}
Метрики для catboost recursive прогноза (30 дней): {'MAE': np.float64(15.852956589987112), 'RMSE': np.float64(26.248367873507334), 'SMAPE': np.float64(0.20709582572138363)}
